In [1]:
# %%
# =============================================================================
# STEP 0: IMPORTS AND GEE INITIALIZATION
# =============================================================================
# Why: We need Google Earth Engine for satellite data access and geemap for
#      visualization. All processing happens on Google's servers — your HPC
#      just sends commands and receives results.

import ee
import geemap
import csv
import json
from datetime import datetime, timedelta

# Initialize GEE — use whichever method works on your system
try:
    ee.Initialize()
    print("✓ GEE initialized successfully")
except Exception:
    try:
        geemap.ee_initialize()
        ee.Initialize()
        print("✓ GEE initialized via geemap")
    except Exception:
        ee.Authenticate()
        ee.Initialize()
        print("✓ GEE initialized after authentication")

✓ GEE initialized successfully


In [2]:
# %%
# =============================================================================
# STEP 1: STUDY AREA AND TIME PERIODS
# =============================================================================

BBOX = [89.8, 23.5, 92.8, 25.5]
roi = ee.Geometry.Rectangle(BBOX)

BASELINE_START = "2025-01-01"
BASELINE_END   = "2025-03-31"
MONITOR_START  = "2025-04-01"
MONITOR_END    = "2025-09-30"

# Z-score threshold
Z_THRESHOLD = -2.5

# Slope mask threshold (degrees)
SLOPE_MAX = 5

# Date window for compositing individual flood maps
# WHY 6 DAYS: Sentinel-1 revisit is ~6 days when combining A+B satellites.
# Using a 6-day window ensures we capture BOTH ascending AND descending passes
# so the entire AOI is covered. Your previous code used 1-day windows, which
# only captured ONE pass → only half the AOI was visible.
COMPOSITE_WINDOW_DAYS = 6

MIN_BASELINE_IMAGES = 5

print(f"Study area: {BBOX}")
print(f"Baseline: {BASELINE_START} to {BASELINE_END}")
print(f"Monitoring: {MONITOR_START} to {MONITOR_END}")
print(f"Z-score threshold: {Z_THRESHOLD}")
print(f"Composite window: {COMPOSITE_WINDOW_DAYS} days")

Study area: [89.8, 23.5, 92.8, 25.5]
Baseline: 2025-01-01 to 2025-03-31
Monitoring: 2025-04-01 to 2025-09-30
Z-score threshold: -2.5
Composite window: 6 days


In [3]:
# %%
# =============================================================================
# STEP 2: CHECK SENTINEL-1 DATA AVAILABILITY
# =============================================================================
# Why: Before doing any analysis, we need to know how many Sentinel-1 images
#      are available over our area. This tells us:
#      - Whether we have enough baseline images for statistics
#      - How many flood-period images we'll get (= temporal resolution)
#      - Which orbit directions (ascending/descending) are available
#
# Sentinel-1 acquires data in IW (Interferometric Wide) mode over land,
# with VV and VH polarization. Over Bangladesh, we typically get an image
# every 12 days from each orbit direction, so ~6 days combined.

def get_s1_collection(roi, start, end, pol="VV"):
    """Get filtered Sentinel-1 GRD collection."""
    return (ee.ImageCollection("COPERNICUS/S1_GRD")
            .filterBounds(roi)
            .filterDate(start, end)
            .filter(ee.Filter.eq("instrumentMode", "IW"))
            .filter(ee.Filter.listContains("transmitterReceiverPolarisation", pol))
            .select(pol))

# Check baseline availability
baseline_col = get_s1_collection(roi, BASELINE_START, BASELINE_END)
n_baseline = baseline_col.size().getInfo()
print(f"\n--- Data Availability ---")
print(f"Baseline images (Jan-Mar 2025): {n_baseline}")

# Check monitoring period availability
monitor_col = get_s1_collection(roi, MONITOR_START, MONITOR_END)
n_monitor = monitor_col.size().getInfo()
print(f"Monitoring images (Apr-Sep 2025): {n_monitor}")

# Get the dates of monitoring images so we know our temporal resolution
def get_image_dates(collection):
    """Extract acquisition dates from an image collection."""
    dates = collection.aggregate_array("system:time_start").getInfo()
    return sorted(set([datetime.utcfromtimestamp(d/1000).strftime("%Y-%m-%d") for d in dates]))

monitor_dates = get_image_dates(monitor_col)
print(f"\nMonitoring dates ({len(monitor_dates)} unique dates):")
for d in monitor_dates:
    print(f"  {d}")

# Check orbit directions
asc = monitor_col.filter(ee.Filter.eq("orbitProperties_pass", "ASCENDING")).size().getInfo()
desc = monitor_col.filter(ee.Filter.eq("orbitProperties_pass", "DESCENDING")).size().getInfo()
print(f"\nAscending: {asc}, Descending: {desc}")

if n_baseline < MIN_BASELINE_IMAGES:
    print(f"\n⚠ WARNING: Only {n_baseline} baseline images. Need at least {MIN_BASELINE_IMAGES}.")
    print("  Consider extending baseline period or using both ASC+DESC orbits.")
else:
    print(f"\n✓ Sufficient baseline images for Z-score computation.")



--- Data Availability ---
Baseline images (Jan-Mar 2025): 92
Monitoring images (Apr-Sep 2025): 175


/tmp/ipykernel_1280900/1164225165.py:39: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  return sorted(set([datetime.utcfromtimestamp(d/1000).strftime("%Y-%m-%d") for d in dates]))



Monitoring dates (71 unique dates):
  2025-04-01
  2025-04-03
  2025-04-04
  2025-04-06
  2025-04-08
  2025-04-10
  2025-04-13
  2025-04-15
  2025-04-16
  2025-04-18
  2025-04-20
  2025-04-22
  2025-04-25
  2025-04-27
  2025-04-30
  2025-05-02
  2025-05-04
  2025-05-07
  2025-05-09
  2025-05-12
  2025-05-14
  2025-05-19
  2025-05-21
  2025-05-24
  2025-05-26
  2025-05-28
  2025-05-31
  2025-06-02
  2025-06-05
  2025-06-07
  2025-06-12
  2025-06-17
  2025-06-24
  2025-06-29
  2025-07-03
  2025-07-06
  2025-07-08
  2025-07-11
  2025-07-13
  2025-07-15
  2025-07-18
  2025-07-20
  2025-07-23
  2025-07-25
  2025-07-27
  2025-07-30
  2025-08-01
  2025-08-04
  2025-08-06
  2025-08-08
  2025-08-11
  2025-08-13
  2025-08-16
  2025-08-18
  2025-08-20
  2025-08-23
  2025-08-25
  2025-08-28
  2025-08-30
  2025-09-01
  2025-09-04
  2025-09-06
  2025-09-09
  2025-09-11
  2025-09-13
  2025-09-16
  2025-09-18
  2025-09-21
  2025-09-23
  2025-09-25
  2025-09-28

Ascending: 92, Descending: 83

✓ Suffic

In [4]:
# %%
# =============================================================================
# STEP 3: COMPUTE BASELINE STATISTICS (MEAN AND STANDARD DEVIATION)
# =============================================================================
# Why: The Z-score method requires knowing what "normal" looks like for each
#      pixel. We compute the mean and standard deviation of VV backscatter
#      during the dry season (Jan-Mar 2025).
#
# How Z-scores work (from DeVries et al. 2020):
#   Z = (current_backscatter - mean_baseline) / std_baseline
#
#   If Z is very negative (e.g., -3), the pixel is MUCH darker than normal
#   → likely flooded (smooth water reflects radar away = very low backscatter)
#
#   If Z is near 0, the pixel looks normal → not flooded
#
# Why this is better than your current -3dB threshold:
#   Your current code compares a flood image to a reference image (delta < -3dB).
#   But the -3dB threshold doesn't adapt to local conditions. A pixel that's
#   always dark (e.g., a river) won't show a big change even when flooded.
#   The Z-score adapts: it asks "is this pixel unusually dark for THIS pixel?"
#
# We compute separate baselines for ascending and descending orbits because
# they look at the ground from different angles, producing different backscatter.

def compute_baseline_stats(roi, start, end, pol="VV"):
    """
    Compute per-pixel mean and std of VV backscatter during baseline period.
    Returns separate stats for ascending and descending orbits.
    """
    base = get_s1_collection(roi, start, end, pol)

    # Apply border noise mask (remove very low values at swath edges)
    base = base.map(lambda img: img.updateMask(img.gt(-30)))

    # Compute stats for ALL orbits combined
    # (simpler and gives more images per pixel for robust stats)
    mean_img = base.mean().rename("baseline_mean")
    std_img = base.reduce(ee.Reducer.stdDev()).rename("baseline_std")

    # Ensure std is not zero (would cause division by zero in Z-score)
    # Replace zero std with a small value (0.5 dB)
    std_img = std_img.where(std_img.lte(0), 0.5)

    n_images = base.count().rename("baseline_count")

    return mean_img, std_img, n_images

print("Computing baseline statistics...")
baseline_mean, baseline_std, baseline_count = compute_baseline_stats(
    roi, BASELINE_START, BASELINE_END
)
print("✓ Baseline mean and std computed")

# Quick diagnostic: check the baseline stats make sense
# For land in Bangladesh, typical VV backscatter is -8 to -15 dB
# For water, it's -18 to -25 dB
sample_point = ee.Geometry.Point([91.0, 24.8])  # A point in the haor region
stats = baseline_mean.reduceRegion(
    reducer=ee.Reducer.first(),
    geometry=sample_point,
    scale=10
).getInfo()
print(f"\nSample baseline mean at (91.0, 24.8): {stats}")

stats_std = baseline_std.reduceRegion(
    reducer=ee.Reducer.first(),
    geometry=sample_point,
    scale=10
).getInfo()
print(f"Sample baseline std at (91.0, 24.8): {stats_std}")


Computing baseline statistics...
✓ Baseline mean and std computed

Sample baseline mean at (91.0, 24.8): {'baseline_mean': -13.009611119801049}
Sample baseline std at (91.0, 24.8): {'baseline_std': 4.029873301998676}


In [6]:
# %%
# =============================================================================
# STEP 4: COMPUTE Z-SCORE FLOOD MASKS FOR EACH MONITORING DATE
# =============================================================================
# Why: This is the core of the time series analysis. For each Sentinel-1 image
#      during the flood season, we compute the Z-score and classify flooded pixels.
#
# What happens for each image:
#   1. Take the VV backscatter image for that date
#   2. Subtract the baseline mean → anomaly (how different from normal?)
#   3. Divide by baseline std → Z-score (how many std devs below normal?)
#   4. If Z < threshold → classify as flooded
#   5. Apply slope mask to remove false positives on hillsides
#
# The result is a stack of binary flood masks — one per S-1 acquisition date.
# Each pixel is either 1 (flooded) or 0 (not flooded) on each date.

def compute_zscore_flood_mask(image, baseline_mean, baseline_std, z_threshold, slope_mask):
    """
    Compute Z-score flood mask for a single Sentinel-1 image.

    Parameters:
    -----------
    image : ee.Image — single S-1 VV backscatter image
    baseline_mean : ee.Image — per-pixel mean from dry season
    baseline_std : ee.Image — per-pixel standard deviation from dry season
    z_threshold : float — Z-score cutoff (e.g., -2.5)
    slope_mask : ee.Image — binary mask (1 = flat terrain, 0 = steep)

    Returns:
    --------
    ee.Image with bands:
      - 'zscore': the Z-score value
      - 'flood': binary flood mask (1=flooded, 0=not flooded)
    """
    # Mask border noise
    masked = image.updateMask(image.gt(-30))

    # Compute Z-score: (observed - mean) / std
    zscore = masked.subtract(baseline_mean).divide(baseline_std).rename("zscore")

    # Classify: Z < threshold = flooded
    flood = zscore.lt(z_threshold).rename("flood")

    # Apply slope mask (only keep flood detections on flat terrain)
    flood = flood.updateMask(slope_mask)

    # Apply focal mode smoothing to remove salt-and-pepper noise
    # radius=60m covers ~6 pixels at 10m resolution
    flood_clean = flood.focal_mode(radius=60, units="meters").rename("flood_clean")

    # Preserve the timestamp
    return flood_clean.copyProperties(image, ["system:time_start"])


# Prepare slope mask from SRTM
slope = ee.Terrain.slope(ee.Image("USGS/SRTMGL1_003"))
slope_mask = slope.lt(SLOPE_MAX)  # 1 where flat, 0 where steep

# Get the monitoring collection
monitor_col = get_s1_collection(roi, MONITOR_START, MONITOR_END)
monitor_col = monitor_col.map(lambda img: img.updateMask(img.gt(-30)))

# Apply Z-score flood detection to each image
print("Computing Z-score flood masks for each monitoring date...")
flood_masks = monitor_col.map(
    lambda img: compute_zscore_flood_mask(
        img, baseline_mean, baseline_std, Z_THRESHOLD, slope_mask
    )
)

n_masks = flood_masks.size().getInfo()
print(f"✓ Generated {n_masks} flood masks")


Computing Z-score flood masks for each monitoring date...
✓ Generated 175 flood masks


In [7]:
# %%
# =============================================================================
# STEP 5: COMPUTE FIRST-WET-DATE MAP AND FLOOD METRICS
# =============================================================================
# Why: The first-wet-date map is YOUR MOST IMPORTANT FIGURE.
#      It shows, for each pixel, the FIRST date that pixel was classified as
#      flooded during the 2025 monsoon season.
#
# If haors are connected:
#   → You'll see a spatial gradient: upstream/river areas flood first (early dates),
#     then flooding spreads outward into adjacent haors (later dates).
#     The map will show a smooth color transition from early (blue) to late (red).
#
# If haors are isolated:
#   → Each haor will flood independently based on local rainfall, not propagation.
#     The map will look patchy — adjacent haors might flood at very different times
#     with no spatial pattern.
#
# Additional metrics we compute:
#   - total_wet_count: number of dates each pixel was wet (= duration proxy)
#   - wet_fraction: proportion of dates wet (= flood frequency)
#   - total_flood_area_per_date: total flooded area across the region per date

def compute_first_wet_date(flood_mask_collection):
    """
    Compute per-pixel first-wet-date from a collection of binary flood masks.

    Logic: For each pixel, find the earliest image date where flood=1.
    We do this by converting each image's timestamp to a "days since epoch"
    band, masking it where flood=0, and taking the minimum across all dates.
    """
    def add_date_band(img):
        """Add a band with the image date as days since 2025-01-01."""
        # Get the date as milliseconds since epoch
        date_ms = img.date().millis()
        # Convert to days since 2025-01-01
        epoch_ms = ee.Date("2025-01-01").millis()
        days = ee.Number(date_ms).subtract(epoch_ms).divide(86400000)  # ms to days
        # Create a constant image with this date value
        date_img = ee.Image.constant(days).float().rename("date_days")
        # Mask: only keep date where flood = 1
        return date_img.updateMask(img.select("flood_clean"))

    # Apply to each flood mask
    date_images = flood_mask_collection.map(add_date_band)

    # First wet date = minimum date across all images (earliest flood detection)
    first_wet = date_images.min().rename("first_wet_date")

    return first_wet


def compute_flood_metrics(flood_mask_collection):
    """
    Compute per-pixel flood duration and frequency metrics.
    """
    # Total number of dates this pixel was classified as wet
    total_wet = flood_mask_collection.select("flood_clean").sum().rename("total_wet_count")

    # Total number of valid observations (not masked)
    total_obs = flood_mask_collection.select("flood_clean").count().rename("total_obs_count")

    # Wet fraction = proportion of dates that were wet
    wet_fraction = total_wet.divide(total_obs).rename("wet_fraction")

    return total_wet, total_obs, wet_fraction


def compute_area_time_series(flood_mask_collection, roi, scale=100):
    """
    Compute total flooded area (km²) for each date in the collection.
    This gives you the flood hydrograph in area-space.

    Parameters:
    -----------
    scale : int — pixel size in meters for area calculation (100m is fast, 10m is precise)
    """
    def get_flood_area(img):
        """Compute flooded area in km² for a single image."""
        # Pixel area in m²
        pixel_area = img.select("flood_clean").multiply(ee.Image.pixelArea())
        # Sum over ROI
        area_m2 = pixel_area.reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=roi,
            scale=scale,
            maxPixels=1e12
        ).get("flood_clean")
        # Convert m² to km²
        area_km2 = ee.Number(area_m2).divide(1e6)
        return img.set("flood_area_km2", area_km2)

    return flood_mask_collection.map(get_flood_area)


print("Computing first-wet-date map...")
first_wet_date = compute_first_wet_date(flood_masks)
print("✓ First-wet-date map computed")

print("\nComputing flood duration and frequency metrics...")
total_wet, total_obs, wet_fraction = compute_flood_metrics(flood_masks)
print("✓ Flood metrics computed")

print("\nComputing area time series (this may take a moment)...")
flood_masks_with_area = compute_area_time_series(flood_masks, roi, scale=100)

# Extract the area values
area_list = flood_masks_with_area.aggregate_array("flood_area_km2").getInfo()
date_list = flood_masks_with_area.aggregate_array("system:time_start").getInfo()

# Convert to readable format
area_dates = []
for d, a in zip(date_list, area_list):
    date_str = datetime.utcfromtimestamp(d/1000).strftime("%Y-%m-%d")
    area_dates.append({"date": date_str, "area_km2": round(a, 1) if a else 0})

print("\n--- Flood Area Time Series ---")
print(f"{'Date':<14} {'Area (km²)':>12}")
print("-" * 28)
for item in sorted(area_dates, key=lambda x: x["date"]):
    print(f"{item['date']:<14} {item['area_km2']:>12.1f}")

print(f"\n✓ Area time series extracted for {len(area_dates)} dates")


Computing first-wet-date map...
✓ First-wet-date map computed

Computing flood duration and frequency metrics...
✓ Flood metrics computed

Computing area time series (this may take a moment)...

--- Flood Area Time Series ---
Date             Area (km²)
----------------------------
2025-04-01             12.8
2025-04-01            143.0
2025-04-01             16.1
2025-04-03              3.6
2025-04-03            180.6
2025-04-03            129.8
2025-04-04              0.1
2025-04-06            305.4
2025-04-06            634.5
2025-04-06              3.0
2025-04-08             16.2
2025-04-08            503.0
2025-04-08             72.9
2025-04-10              0.0
2025-04-13             19.4
2025-04-13            231.8
2025-04-13              4.8
2025-04-15              0.7
2025-04-15            176.1
2025-04-15            212.8
2025-04-16              0.0
2025-04-18             96.0
2025-04-18            444.9
2025-04-18              0.5
2025-04-20              4.7
2025-04-20       

/tmp/ipykernel_1280900/44076093.py:114: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  date_str = datetime.utcfromtimestamp(d/1000).strftime("%Y-%m-%d")


In [8]:
# %%
# =============================================================================
# STEP 6: VISUALIZE — FIRST-WET-DATE MAP WITH SMOOTH GRADIENT
# =============================================================================
# CHANGED: Now uses a 30-level smooth color gradient instead of 6 monthly colors.
#
# Why this matters:
#   With 6 colors, most of the haor region showed up as one uniform green
#   (= June), making it impossible to see propagation within that month.
#   With 30+ levels, you can distinguish flooding that happened on June 1
#   vs June 12 vs June 20 — which is exactly the temporal resolution
#   you need for propagation analysis.
#
# Color scheme: a perceptually uniform gradient from dark blue (earliest)
# through cyan, green, yellow, orange to dark red (latest).
# This is similar to the 'turbo' colormap used in scientific visualization.

print("\nCreating main analysis map...")
m = geemap.Map()
m.centerObject(roi, 8)

# --- CHANGED: Smooth 30-level color gradient for first-wet-date ---
# Day 90 = April 1, Day 272 = September 30
# Each "step" in the palette ≈ 6 days (matching S-1 revisit cycle)
first_wet_vis = {
    "min": 90,
    "max": 272,
    "palette": [
        # April (days 90-120): deep blue → blue
        "023858", "045a8d", "0570b0", "3690c0",
        # May (days 121-151): cyan → teal
        "74a9cf", "66c2a4", "41ae76", "238b45",
        # June (days 152-181): green → yellow-green
        "006d2c", "31a354", "78c679", "addd8e",
        # July (days 182-212): yellow → yellow-orange
        "d9f0a3", "f7fcb1", "ffeda0", "fed976",
        # August (days 213-243): orange
        "feb24c", "fd8d3c", "fc4e2a", "e31a1c",
        # September (days 244-272): red → dark red
        "d7301f", "bd0026", "a50026", "800026",
    ]
}

m.addLayer(
    first_wet_date.clip(roi),
    first_wet_vis,
    "First Wet Date (smooth gradient)",
    True
)

# Duration
duration_vis = {
    "min": 0,
    "max": 20,
    "palette": ["FFFFFF", "C6DBEF", "6BAED6", "2171B5", "08306B"]
}
m.addLayer(total_wet.clip(roi), duration_vis, "Total Wet Count (duration)", False)

# Frequency
frequency_vis = {
    "min": 0,
    "max": 1,
    "palette": ["FFFFFF", "FED976", "FEB24C", "FD8D3C", "FC4E2A", "E31A1C", "B10026"]
}
m.addLayer(wet_fraction.clip(roi), frequency_vis, "Wet Fraction (frequency)", False)

# Baseline mean for context
m.addLayer(baseline_mean.clip(roi), {"min": -20, "max": -5}, "Baseline Mean VV (dB)", False)

# ROI outline
m.addLayer(ee.Image().paint(roi, 1, 2), {"palette": ["FFFFFF"]}, "Study Area", True)

print("✓ Main map created — run 'm' to display")
print("\nColor guide for First Wet Date:")
print("  Dark blue  = April (earliest flooding)")
print("  Cyan/teal  = May")
print("  Green      = Early June")
print("  Yellow-grn = Late June")
print("  Yellow     = July")
print("  Orange     = August")
print("  Red        = September (latest flooding)")



Creating main analysis map...
✓ Main map created — run 'm' to display

Color guide for First Wet Date:
  Dark blue  = April (earliest flooding)
  Cyan/teal  = May
  Green      = Early June
  Yellow-grn = Late June
  Yellow     = July
  Orange     = August
  Red        = September (latest flooding)


In [10]:
m

Map(center=[24.504754997740452, 91.30000000000003], controls=(WidgetControl(options=['position', 'transparent_…

In [9]:
# %%
# =============================================================================
# STEP 7: INDIVIDUAL DATE FLOOD MAPS — FIXED FOR FULL COVERAGE
# =============================================================================
# CHANGED: Two fixes for the half-empty map problem:
#
# Fix 1: Use 6-day composite windows instead of 1-day windows.
#   Sentinel-1 passes your area from different directions on different days.
#   On any single day, only ONE orbital track is active → covers ~half your AOI.
#   By compositing over 6 days, we capture BOTH ascending and descending passes,
#   giving full spatial coverage of the entire study area.
#
# Fix 2: Generate composites at regular 6-day intervals.
#   Instead of picking specific dates (which might miss some), we create
#   a regular grid of 6-day windows from April 1 to September 30.
#   This gives ~30 time steps — one per S-1 revisit cycle.

def generate_composite_dates(start_date, end_date, interval_days):
    """Generate a list of start dates for composite windows."""
    dates = []
    current = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")
    while current < end:
        dates.append(current.strftime("%Y-%m-%d"))
        current += timedelta(days=interval_days)
    return dates

def create_composite_flood_map(flood_masks, start_date, window_days):
    """
    Create a composite flood map over a time window.
    Uses .max() to take the union of all flood detections in the window.
    If ANY image in the window shows flooding at a pixel, it's marked as flooded.
    """
    end_date = (datetime.strptime(start_date, "%Y-%m-%d") + 
                timedelta(days=window_days)).strftime("%Y-%m-%d")
    
    window_masks = flood_masks.filterDate(start_date, end_date)
    # Use .max() → pixel is flooded if it was flooded in ANY image in the window
    composite = window_masks.select("flood_clean").max().rename("flood_composite")
    return composite, window_masks.size()


# Generate regular 6-day composite dates
composite_dates = generate_composite_dates(MONITOR_START, MONITOR_END, COMPOSITE_WINDOW_DAYS)
print(f"\n--- Generating {len(composite_dates)} composite flood maps ---")
print(f"  Window size: {COMPOSITE_WINDOW_DAYS} days per composite")

# Create the date browser map
print("\nCreating date browser map with full-coverage composites...")
m2 = geemap.Map()
m2.centerObject(roi, 8)

# We'll add every composite as a toggleable layer
# But to avoid overwhelming the layer panel, we group by month
# and only make the first of each month visible by default
composites_added = 0

for date_str in composite_dates:
    end_dt = datetime.strptime(date_str, "%Y-%m-%d") + timedelta(days=COMPOSITE_WINDOW_DAYS)
    end_str = end_dt.strftime("%Y-%m-%d")
    
    # Create composite
    window_masks = flood_masks.filterDate(date_str, end_str)
    n_imgs = window_masks.size().getInfo()
    
    if n_imgs > 0:
        composite = window_masks.select("flood_clean").max()
        
        # Determine month for color coding
        month = datetime.strptime(date_str, "%Y-%m-%d").month
        month_colors = {
            4: "0570b0",  # April: blue
            5: "41ae76",  # May: teal
            6: "78c679",  # June: green
            7: "fed976",  # July: yellow
            8: "fc4e2a",  # August: orange
            9: "bd0026",  # September: red
        }
        color = month_colors.get(month, "FF0000")
        
        # Make first composite of each month visible by default
        day = datetime.strptime(date_str, "%Y-%m-%d").day
        visible = (day <= COMPOSITE_WINDOW_DAYS)  # First composite of month
        
        m2.addLayer(
            composite.selfMask().clip(roi),
            {"palette": [color]},
            f"{date_str} to {end_str} (n={n_imgs})",
            visible
        )
        composites_added += 1

# Add first-wet-date as reference layer
m2.addLayer(first_wet_date.clip(roi), first_wet_vis, "First Wet Date", False)

# Add ROI
m2.addLayer(ee.Image().paint(roi, 1, 2), {"palette": ["FFFFFF"]}, "Study Area", True)

print(f"✓ Added {composites_added} composite flood maps")
print("  Each layer covers a 6-day window with full AOI coverage")
print("  Toggle layers on/off in the panel to see flood progression")
print("  Colors: blue=Apr, teal=May, green=Jun, yellow=Jul, orange=Aug, red=Sep")
print("\n  Display with 'm2'")



--- Generating 31 composite flood maps ---
  Window size: 6 days per composite

Creating date browser map with full-coverage composites...
✓ Added 30 composite flood maps
  Each layer covers a 6-day window with full AOI coverage
  Toggle layers on/off in the panel to see flood progression
  Colors: blue=Apr, teal=May, green=Jun, yellow=Jul, orange=Aug, red=Sep

  Display with 'm2'


In [11]:
m2

Map(center=[24.504754997740452, 91.30000000000003], controls=(WidgetControl(options=['position', 'transparent_…

In [12]:
# %%
# =============================================================================
# STEP 8: SIDE-BY-SIDE COMPARISON MAP FOR KEY DATES
# =============================================================================
# NEW: Creates a focused comparison of pre-flood, peak-flood, and post-peak.
# This is useful for your D-Lab presentation — shows the dramatic change.

print("\nCreating key-dates comparison map...")
m3 = geemap.Map()
m3.centerObject(roi, 8)

# Pre-flood: April (before significant flooding)
pre_flood = flood_masks.filterDate("2025-04-01", "2025-04-30")
n_pre = pre_flood.size().getInfo()
if n_pre > 0:
    pre_composite = pre_flood.select("flood_clean").max()
    m3.addLayer(pre_composite.selfMask().clip(roi), 
                {"palette": ["0570b0"]}, f"Pre-flood: April (n={n_pre})", True)

# Peak flood: late May to early June (confirmed peak event)
peak_flood = flood_masks.filterDate("2025-05-25", "2025-06-10")
n_peak = peak_flood.size().getInfo()
if n_peak > 0:
    peak_composite = peak_flood.select("flood_clean").max()
    m3.addLayer(peak_composite.selfMask().clip(roi),
                {"palette": ["FF0000"]}, f"Peak flood: 25 May-10 Jun (n={n_peak})", True)

# Post-peak: July
post_peak = flood_masks.filterDate("2025-07-01", "2025-07-31")
n_post = post_peak.size().getInfo()
if n_post > 0:
    post_composite = post_peak.select("flood_clean").max()
    m3.addLayer(post_composite.selfMask().clip(roi),
                {"palette": ["FFA500"]}, f"Post-peak: July (n={n_post})", False)

# Late monsoon: September
late_monsoon = flood_masks.filterDate("2025-09-01", "2025-09-30")
n_late = late_monsoon.size().getInfo()
if n_late > 0:
    late_composite = late_monsoon.select("flood_clean").max()
    m3.addLayer(late_composite.selfMask().clip(roi),
                {"palette": ["800026"]}, f"Late monsoon: Sep (n={n_late})", False)

m3.addLayer(ee.Image().paint(roi, 1, 2), {"palette": ["FFFFFF"]}, "Study Area", True)

print("✓ Key-dates comparison map created — run 'm3'")
print("  Blue = pre-flood (April)")
print("  Red = peak flood (late May - early June)")
print("  Orange = post-peak (July)")
print("  Dark red = late monsoon (September)")


Creating key-dates comparison map...
✓ Key-dates comparison map created — run 'm3'
  Blue = pre-flood (April)
  Red = peak flood (late May - early June)
  Orange = post-peak (July)
  Dark red = late monsoon (September)


In [13]:
m3

Map(center=[24.504754997740452, 91.30000000000003], controls=(WidgetControl(options=['position', 'transparent_…

In [27]:
# %%
# =============================================================================
# STEP 9: SAVE AREA TIME SERIES TO CSV
# =============================================================================
# UNCHANGED from v1 — this was working correctly.

csv_filename = "haor_flood_area_timeseries_2025.csv"
area_dates_sorted = sorted(area_dates, key=lambda x: x["date"])

with open(csv_filename, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["date", "area_km2"])
    writer.writeheader()
    writer.writerows(area_dates_sorted)

print(f"✓ Area time series saved to {csv_filename}")

areas = [x["area_km2"] for x in area_dates_sorted if x["area_km2"] > 0]
if areas:
    max_area = max(areas)
    max_idx = [x["area_km2"] for x in area_dates_sorted].index(max_area)
    print(f"\n--- Flood Area Summary ---")
    print(f"  Observation dates: {len(area_dates_sorted)}")
    print(f"  Dates with flooding: {len(areas)}")
    print(f"  Maximum flooded area: {max_area:.1f} km²")
    print(f"  Date of maximum: {area_dates_sorted[max_idx]['date']}")
    print(f"  Min flooded area (>0): {min(areas):.1f} km²")


✓ Area time series saved to haor_flood_area_timeseries_2025.csv

--- Flood Area Summary ---
  Observation dates: 175
  Dates with flooding: 162
  Maximum flooded area: 6269.4 km²
  Date of maximum: 2025-06-02
  Min flooded area (>0): 0.1 km²


In [34]:
# %%
# =============================================================================
# STEP 10: EXPORT RESULTS
# =============================================================================
# UNCHANGED from v1.

EXPORT_SCALE = 100
EXPORT_CRS = "EPSG:4326"

task_fwd = ee.batch.Export.image.toDrive(
    image=first_wet_date.clip(roi).float(),
    description="haor_first_wet_date_2025",
    folder="haor_flood_analysis",
    fileNamePrefix="first_wet_date_2025",
    region=roi,
    scale=EXPORT_SCALE,
    crs=EXPORT_CRS,
    maxPixels=1e12
)
task_fwd.start()

task_wf = ee.batch.Export.image.toDrive(
    image=wet_fraction.clip(roi).float(),
    description="haor_wet_fraction_2025",
    folder="haor_flood_analysis",
    fileNamePrefix="wet_fraction_2025",
    region=roi,
    scale=EXPORT_SCALE,
    crs=EXPORT_CRS,
    maxPixels=1e12
)
task_wf.start()

task_twc = ee.batch.Export.image.toDrive(
    image=total_wet.clip(roi).int16(),
    description="haor_total_wet_count_2025",
    folder="haor_flood_analysis",
    fileNamePrefix="total_wet_count_2025",
    region=roi,
    scale=EXPORT_SCALE,
    crs=EXPORT_CRS,
    maxPixels=1e12
)
task_twc.start()

print("\n--- Export Tasks ---")
print("Uncomment task.start() lines above to begin exports.")
print("Files will appear in Google Drive → 'haor_flood_analysis' folder")



--- Export Tasks ---
Uncomment task.start() lines above to begin exports.
Files will appear in Google Drive → 'haor_flood_analysis' folder
